# 🚀 Zalo AI Challenge 2026 - DataMate: QLoRA Fine-Tuning + Bộ Đọc Báo Lọc Báo (14.7B SLM)
Notebook này tích hợp **3 cải tiến cốt lõi** giúp tối đa hóa điểm **Penalized Accuracy (PA)** (+1 đúng, -1 sai, 0 abstain):
1. **Bộ Đọc Báo Lọc Báo 3 Tầng (`DocumentSchemaFilter`) & Bộ Khử Nhiễu Telex**: Tự động đọc `README.md`, kiểm tra khoảng thời gian dữ liệu `[min_date, max_date]`, kiểm tra schema bảng thực tế và chặn hoàn toàn hiện tượng SLM 14B ép câu hỏi ngoài phạm vi vào `revenue_total` / `best_sellers`.
2. **Fine-Tuning QLoRA 4-bit NF4 (`train_qlora.py`)**: Huấn luyện trực tiếp `Qwen/Qwen2.5-14B-Instruct` (14.7B tham số, đạt chuẩn $\le 15\text{B}$ của BTC) trên 3.000 mẫu SFT tiếng Việt (bao gồm cả từ lóng, viết tắt, sai chính tả và kỹ thuật *Calibrated Selective Abstention*).
3. **Deterministic Symbolic Execution Engine**: Đảm bảo tính toán số học chính xác tuyệt đối đến từng đồng VND và làm tròn đúng Mục 9 Rulebook.

In [ ]:
# @title Bước 1: Cài đặt thư viện QLoRA & Kiểm tra GPU (Colab L4 / T4)
# Không nâng cấp torch và giữ chuẩn pandas==2.2.3 của Colab để tránh xung đột torchaudio / pandas 3.x
!pip uninstall -y -q torchaudio
!pip install -q "pandas==2.2.3" bitsandbytes peft accelerate openpyxl
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

In [ ]:
# @title Bước 2: Giải nén zalo_colab_package.zip (Thư mục chuẩn /content/code không dấu)
# Hãy upload file zalo_colab_package.zip lên thư mục /content/ (hoặc Google Drive) trước khi chạy ô này
import os
if os.path.exists('/content/zalo_colab_package.zip'):
    !unzip -q -o /content/zalo_colab_package.zip -d /content/
    print('Đã giải nén thành công zalo_colab_package.zip!')
else:
    print('Không tìm thấy /content/zalo_colab_package.zip, vui lòng kiểm tra lại đường dẫn.')

%cd /content/code
!ls -la

In [ ]:
# @title Bước 3: Chạy Fine-Tuning QLoRA trên tập dữ liệu tiếng Việt SFT (Tùy chọn: 14B hoặc 7B)
# Mặc định dùng Qwen/Qwen2.5-14B-Instruct (14.7B tham số, chuẩn <= 15B).
# Nếu muốn train nhanh hơn (~5-8 phút trên L4), bạn có thể đổi BASE_MODEL thành 'Qwen/Qwen2.5-7B-Instruct'.
BASE_MODEL = 'Qwen/Qwen2.5-14B-Instruct'  # @param ['Qwen/Qwen2.5-14B-Instruct', 'Qwen/Qwen2.5-7B-Instruct']
EPOCHS = 1.0  # @param {type:'number'}

!python train_qlora.py --base_model "$BASE_MODEL" --train_file sft_train_data.jsonl --output_dir qlora_adapter --epochs $EPOCHS --batch_size 4 --grad_accum 4

In [ ]:
# @title Bước 4: Chạy Inference 1,000 câu Public Test (Kết hợp QLoRA Adapter + Bộ Đọc Báo Lọc Báo 3 Tầng)
import json
BASE_MODEL = 'Qwen/Qwen2.5-14B-Instruct'  # @param ['Qwen/Qwen2.5-14B-Instruct', 'Qwen/Qwen2.5-7B-Instruct']

!python run_local.py \
    --split public_test \
    --use_slm \
    --slm_provider transformers \
    --slm_model "$BASE_MODEL" \
    --lora_adapter qlora_adapter \
    --output /content/submission_14b_qlora.jsonl

# Kiểm tra tính hợp lệ tuyệt đối theo Rulebook Mục 9 trước khi tải về
sub_path = '/content/submission_14b_qlora.jsonl'
with open(sub_path, 'r', encoding='utf-8') as f:
    lines = [json.loads(line) for line in f if line.strip()]

type_counts = {}
for item in lines:
    assert list(item.keys())[0] == 'id', f'Lỗi thứ tự key: {item}'
    t = item['type']
    type_counts[t] = type_counts.get(t, 0) + 1

print(f'\n✅ Kiểm tra hoàn tất: {len(lines)}/1000 dòng hợp lệ 100% theo Rulebook Mục 9!')
print('Phân bố các loại câu trả lời:', json.dumps(type_counts, ensure_ascii=False, indent=2))

try:
    from google.colab import files
    files.download(sub_path)
except Exception:
    print(f'File đã lưu tại: {sub_path}')